<a href="https://colab.research.google.com/github/PranjalAsthana/InfoRetrievalMethods/blob/main/SinglePassInMemoryIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import pickle
from collections import defaultdict

In [9]:
class SPIMIIndexer:
  def __init__(self, output_dir='spimi_blocks', max_block_size=10000):
    self.output_dir = output_dir
    self.max_block_size = max_block_size
    os.makedirs(self.output_dir, exist_ok=True)
    self.block_counter = 0

  def spimi_invert(self, token_stream):
    """
        token_stream: generator or iterable yielding tuples of (term, doc_id)
        """
    block_files = []
    # dictionary: term -> list of doc_ids (postings list)
    dictionary = defaultdict(list)
    current_postings_count = 0

    for term, doc_id in token_stream:
      postings_list = dictionary[term]

      # Add doc_id if it's not a duplicate for this term in this block
      if not postings_list or postings_list[-1] != doc_id:
        postings_list.append(doc_id)
        current_postings_count += 1

      # Check if memory/block capacity is full
      if self._is_full(current_postings_count):
        block_file = self._write_block_to_disk(dictionary)
        block_files.append(block_file)
        dictionary = defaultdict(list)
        current_postings_count = 0

    # Write any remaining terms in the dictionary as a final block

    if dictionary:
      block_file = self._write_block_to_disk(dictionary)
      block_files.append(block_file)

    return block_files

  def _is_full(self, count):
    return count >= self.max_block_size

  def _write_block_to_disk(self, dictionary):
    self.block_counter += 1
    block_filename = os.path.join(
        self.output_dir, f'block_{self.block_counter}.bin'
    )

    # Sort terms alphabetically before writing to disk
    sorted_terms = sorted(dictionary.keys())
    sorted_dict = {term: dictionary[term] for term in sorted_terms}

    with open(block_filename, 'wb') as f:
      pickle.dump(sorted_dict, f)

    return block_filename

In [11]:
# --- Example Usage ---
def dummy_token_stream():
  # Simulating a stream of (term, doc_id) tuples from a tokenizer
  tokens = [
      ('information', 1),
      ('retrieval', 1),
      ('spimi', 1),
      ('python', 2),
      ('information', 2),
      ('indexing', 2),
  ]
  for t in tokens:
    yield t


if __name__ == '__main__':
  indexer = SPIMIIndexer(output_dir='blocks', max_block_size=5)
  written_blocks = indexer.spimi_invert(dummy_token_stream())
  print('Generated block files:', written_blocks)

  print('\n--- Deserialized Block Contents ---')
  for block_path in written_blocks:
    with open(block_path, 'rb') as f:
      block_data = pickle.load(f)
      print(f'\nFile: {block_path}')
      for term, postings in block_data.items():
        print(f'  "{term}" -> {postings}')

Generated block files: ['blocks/block_1.bin', 'blocks/block_2.bin']

--- Deserialized Block Contents ---

File: blocks/block_1.bin
  "information" -> [1, 2]
  "python" -> [2]
  "retrieval" -> [1]
  "spimi" -> [1]

File: blocks/block_2.bin
  "indexing" -> [2]
